# Classifier that predicts Ticket type
we will use combine_text from the dataset for the main classification traiining 
- sentence-transformer encodes it into a 384-dimensional vector
- that vector goes into a classifier (like logisitc regression or random forest)
- output is then given eg:1,2,3,4 as per the ticket type

this is transfer learning. Here we are BERT's pre trained language understanding, without training it ourself as training requires GPU and hourse of training

In [110]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

In [111]:
## loading the dataset
df = pd.read_csv("../Dataset/cleaned_tickets.csv")

In [112]:
## checking the dataset is correct
print(df.shape)
print(df['Ticket Type'].value_counts())
print(df['ticket_type_encoded'].value_counts())
print(df['combine_text'].iloc[0])

(4500, 9)
Ticket Type
Technical Issue    1072
Product Inquiry     887
Account Access      874
Refund Request      867
Billing Inquiry     800
Name: count, dtype: int64
ticket_type_encoded
4    1072
2     887
0     874
3     867
1     800
Name: count, dtype: int64
something went wrong after the latest update i've had my massage gun pro for a while now but recently something went wrong. there's a clicking sound coming from inside the unit when it's running. nothing i've tried online has worked.


In [113]:
## preparing input and output
X = df['combine_text']
y = df['ticket_type_encoded']

print(X.head())
print(y.head())

0    something went wrong after the latest update i...
1    i have a few questions about a product before ...
2    something went wrong after the latest update i...
3    product question from a potential customer i'm...
4    help with processing a refund i'll keep this b...
Name: combine_text, dtype: str
0    4
1    2
2    4
3    2
4    3
Name: ticket_type_encoded, dtype: int64


In [114]:
## train, test splitting

X_train,X_test, y_train,y_test = train_test_split(X,y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)

(3600,)
(900,)


In [115]:
## generating BERT embeddinhs

print("loading sentence transformer model..")
encoder = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded")

loading sentence transformer model..


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3855.02it/s]


Model loaded


In [116]:
print("Encoding the training data...")
X_train_embeddings = encoder.encode(
    X_train.tolist(),
    show_progress_bar = True,
    batch_size = 32
)

print("encoding test data")
X_test_embeddings  = encoder.encode(
    X_test.tolist(),
    show_progress_bar=True,
    batch_size=32
)

print("train embeddings shape: ", X_train_embeddings.shape)
print("test_embeddings shape ",X_test_embeddings.shape)

Encoding the training data...


Batches: 100%|██████████| 113/113 [00:34<00:00,  3.26it/s]


encoding test data


Batches: 100%|██████████| 29/29 [00:08<00:00,  3.35it/s]

train embeddings shape:  (3600, 384)
test_embeddings shape  (900, 384)


### insight into what sentence transformer really does

- When BERT reads a sentence like "my account is not working", it doesn't just count words like TF-IDF. It reads the entire sentence bidirectionally and captures the meaning of the sentence as a whole.
That meaning gets compressed into 384 numbers. Each number captures some aspect of the semantic content — things like:

   - Is this sentence about a technical problem?
   - Is there urgency in the tone?
   - Is this about account access or billing?

- You don't control what each of those 384 numbers means — BERT learned them during pre-training on billions of sentences. You just use the output.
The key insight: two sentences with similar meanings will have similar 384-dimensional vectors, even if they use completely different words. That's what makes it powerful for your classifier.

- For example:
   - "app keeps crashing" → vector [0.2, -0.5, 0.8, ...]
   - "software won't open" → vector [0.19, -0.48, 0.79, ...]

These vectors will be close to each other. TF-IDF would see completely different words. BERT sees similar meaning.


In [117]:
## training the classifier to perict the ticket type

model1 = LogisticRegression(max_iter=1000)
model1.fit(X_train_embeddings, y_train)
print("model trained")

model trained


With TF-IDF:

- fit learns the vocabulary from training data
- transform applies that learned vocabulary
- You must fit only on training data because the vocabulary is learned

With Sentence Transformers:

- There is no fit step
- The model is already pre-trained on billions of sentences
- It's not learning anything from your data — it's just converting text to vectors using knowledge it already has
- So calling encoder.encode() on both train and test independently is completely fine — no data leakage

In [118]:
le_type = joblib.load("../models/le_type.pkl")   # replace with your actual file path

# Inspect the contents
print(type("le_type"))
print(le_type)

<class 'str'>
LabelEncoder()


In [119]:
## prediction
y_pred = model1.predict(X_test_embeddings)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\n")
print(classification_report(y_test, y_pred, target_names=le_type.classes_))

Accuracy: 0.9766666666666667


                 precision    recall  f1-score   support

 Account Access       0.99      0.99      0.99       171
Billing Inquiry       0.97      0.97      0.97       150
Product Inquiry       0.98      0.96      0.97       163
 Refund Request       0.96      0.97      0.96       164
Technical Issue       0.98      0.99      0.98       252

       accuracy                           0.98       900
      macro avg       0.98      0.97      0.98       900
   weighted avg       0.98      0.98      0.98       900



In [120]:
model2 = RandomForestClassifier()
model2.fit(X_train_embeddings,y_train)
print("model trained")

model trained


In [121]:
y_pred2 = model2.predict(X_test_embeddings)

print("Accuracy:", accuracy_score(y_test, y_pred2))
print("\n")
print(classification_report(y_test, y_pred2, target_names=le_type.classes_))

Accuracy: 0.9611111111111111


                 precision    recall  f1-score   support

 Account Access       0.99      0.97      0.98       171
Billing Inquiry       0.96      0.95      0.96       150
Product Inquiry       0.96      0.94      0.95       163
 Refund Request       0.95      0.96      0.96       164
Technical Issue       0.95      0.97      0.96       252

       accuracy                           0.96       900
      macro avg       0.96      0.96      0.96       900
   weighted avg       0.96      0.96      0.96       900



In [123]:
print("Any overlap?", len(set(X_train.tolist()) & set(X_test.tolist())))
print("Total rows:", len(df))
print("Unique combine_text:", df['combine_text'].nunique())
print("Duplicates:", len(df) - df['combine_text'].nunique())

Any overlap? 1
Total rows: 4500
Unique combine_text: 4494
Duplicates: 6


### Insights from model training

- logistic regression gave an overall accuracy of 97.6%
- Random forest gave an overall accuracy of 96.1%
- Logistic regression gives a btter and more consistent recall for all the ticket types
- we have 59% clear ticktes with good ticket description and subject
- 30% of the tickets are made ambiguous intentionally to make it nearer to real life scenarios where the words are not clearly indicating and  particular ticket type or we have significant noise in the description
- Vague tickets (~7%) — almost no category signal. Things like "something isn't right, please help." Genuinely unclassifiable.
- Typo/informal tickets (~20%, overlapping with above) — realistic misspellings (recieved, havn't, dosent, somthing), run-on sentences, lowercase throughout, no punctuation.
- Tone-mismatched tickets (~8%)

- still the bert sentence transformer's ability to manage messy texts with high noise helps in giving such high accuracy

- The model can still be confused between Billing vs Refund tickets and Technical vs Account Access tickets

In [124]:
joblib.dump(model1, '../models/type_classifier.pkl')
joblib.dump(encoder, '../models/sentence_encoder.pkl')
print("Model Saved.")

Model Saved.
